# TUJI1 — preparación y clustering autocontenidos


## 2. Importaciones y rutas


In [10]:
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler

try:
    from IPython.display import display
except ImportError:
    display = print


# Si Jupyter se inicia fuera de la carpeta del paquete, escribe aquí su ruta.
# Ejemplo: PROJECT_ROOT = Path(r"C:/TFM/notebooks_autocontenidos_sin_core")
PROJECT_ROOT = None

cwd = Path.cwd().resolve()
root_candidates = [cwd, cwd.parent, cwd.parent.parent]
if PROJECT_ROOT is not None:
    ROOT = Path(PROJECT_ROOT).expanduser().resolve()
else:
    ROOT = next(
        (
            candidate
            for candidate in root_candidates
            if sum((candidate / name).is_dir() for name in ["TUT", "TUJI1", "UJIIndoor", "SOD"])
            >= 2
        ),
        cwd,
    )

SEED = 42
TARGET_COLUMNS = ["TARGET_X_M", "TARGET_Y_M"]
print("Raíz utilizada:", ROOT)


Raíz utilizada: /home/coder/Indoor/Notebooks


## 3. Lectura y preprocesado RSSI

Se normalizan los nombres de columnas, se detectan automáticamente WAP/MAC
y el escalador se ajusta solo con la partición de entrenamiento.


In [11]:
def natural_key(text: str) -> List[object]:
    return [int(piece) if piece.isdigit() else piece for piece in re.split(r"(\d+)", text)]


def find_file_case_insensitive(filename: str, directories: Sequence[Path]) -> Path:
    """Busca un nombre sin depender de mayusculas/minusculas."""
    checked: List[str] = []
    target = filename.casefold()
    for directory in directories:
        directory = Path(directory)
        checked.append(str(directory / filename))
        if not directory.exists():
            continue
        direct = directory / filename
        if direct.exists():
            return direct.resolve()
        for child in directory.iterdir():
            if child.is_file() and child.name.casefold() == target:
                return child.resolve()
    raise FileNotFoundError(
        f"No se encontro {filename}. Rutas comprobadas:\n- " + "\n- ".join(checked)
    )


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [str(c).strip().upper() for c in out.columns]
    return out


def detect_rssi_columns(df: pd.DataFrame) -> List[str]:
    cols = [c for c in df.columns if re.fullmatch(r"(?:WAP|MAC)\d+", str(c).upper())]
    cols = sorted(cols, key=natural_key)
    if not cols:
        raise ValueError("No se detectaron columnas RSSI WAPnnn o MACnnn.")
    return cols


class RSSIPreprocessor:
    """Imputa ausencias, estandariza RSSI con train y anade mascara de deteccion."""

    def __init__(self, missing_value: float = 100.0, fill_value: float = -110.0, use_mask: bool = True):
        self.missing_value = float(missing_value)
        self.fill_value = float(fill_value)
        self.use_mask = bool(use_mask)
        self.scaler = StandardScaler()
        self.columns: List[str] = []

    def _clean(self, df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
        raw = df[self.columns].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
        observed = np.isfinite(raw) & (raw != self.missing_value)
        clean = np.where(observed, raw, self.fill_value).astype(np.float32)
        return clean, observed.astype(np.float32)

    def fit(self, df: pd.DataFrame, columns: Sequence[str]) -> "RSSIPreprocessor":
        self.columns = list(columns)
        clean, _ = self._clean(df)
        self.scaler.fit(clean)
        return self

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        clean, mask = self._clean(df)
        scaled = self.scaler.transform(clean).astype(np.float32)
        if self.use_mask:
            return np.concatenate([scaled, mask], axis=1).astype(np.float32)
        return scaled

    def fit_transform(self, df: pd.DataFrame, columns: Sequence[str]) -> np.ndarray:
        return self.fit(df, columns).transform(df)


## 4. Coordenadas y división de posiciones


In [12]:
def position_key(df: pd.DataFrame, columns: Sequence[str]) -> pd.Series:
    return df[list(columns)].astype(str).agg("|".join, axis=1)


### 4.1. Adaptación específica de TUJI1


In [13]:
def _try_find(filename: str, directories: Sequence[Path]) -> Optional[Path]:
    try:
        return find_file_case_insensitive(filename, directories)
    except FileNotFoundError:
        return None


def _collapse_cluster_long(df: pd.DataFrame) -> pd.DataFrame:
    """Elimina la repeticion artificial de una fila para cada valor de K."""
    out = normalize_columns(df)
    out = out.drop(columns=["CLUSTER", "N_CLUSTERS", "SPLIT"], errors="ignore")
    if "ID" in out.columns and out["ID"].notna().all():
        out = out.drop_duplicates(subset=["ID"], keep="first")
    else:
        out = out.drop_duplicates(keep="first")
    return out.reset_index(drop=True)


def _add_local_metric_targets(
    df: pd.DataFrame,
    x_column: str = "POS_X",
    y_column: str = "POS_Y",
    crs_name: str = "LOCAL_METRES",
) -> pd.DataFrame:
    out = df.copy()
    out["TARGET_X_M"] = pd.to_numeric(out[x_column], errors="raise").astype(float)
    out["TARGET_Y_M"] = pd.to_numeric(out[y_column], errors="raise").astype(float)
    out["METRIC_CRS"] = crs_name
    return out


def _normalise_tuji_source(df: pd.DataFrame) -> pd.DataFrame:
    out = normalize_columns(df)
    rename = {
        "COORD_COL_0": "POS_X",
        "COORD_COL_1": "POS_Y",
        "COORD_COL_2": "POS_Z",
    }
    out = out.rename(columns={k: v for k, v in rename.items() if k in out.columns})
    rss_columns = [c for c in out.columns if c.startswith("RSS_COL_")]
    if rss_columns:
        rss_columns = sorted(rss_columns, key=lambda c: int(c.rsplit("_", 1)[1]))
        out = out.rename(columns={c: f"WAP{i + 1:03d}" for i, c in enumerate(rss_columns)})
    return out


def _load_tuji_split(
    raw_name: str,
    fallback_name: str,
    directories: Sequence[Path],
) -> Tuple[pd.DataFrame, Dict[str, str]]:
    raw_path = _try_find(raw_name, directories)
    if raw_path is not None:
        return _normalise_tuji_source(pd.read_csv(raw_path)), {
            "method": "original_file",
            "file": str(raw_path),
        }
    fallback = find_file_case_insensitive(fallback_name, directories)
    return _normalise_tuji_source(_collapse_cluster_long(pd.read_csv(fallback))), {
        "method": "collapsed_clusterized_file",
        "file": str(fallback),
    }


## 5. Guardado y router RSSI→clúster

Los identificadores de fila permiten unir cada partición con sus rutas.
KMeans se ajusta con las posiciones de train; un Extra Trees aprende a
reproducir esas zonas desde RSSI para validación y test.


### 5.1. Guardado de las tres particiones


In [14]:
def _assign_row_ids(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    out = df.reset_index(drop=True).copy()
    out.insert(0, "ROW_ID", [f"{prefix}_{i:07d}" for i in range(len(out))])
    return out


def save_base_splits(
    train: pd.DataFrame,
    val: pd.DataFrame,
    test: pd.DataFrame,
    output_dir: Path,
    prefix: str,
) -> Dict[str, Path]:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    split_frames = {
        "train": _assign_row_ids(train, f"{prefix}_train"),
        "val": _assign_row_ids(val, f"{prefix}_val"),
        "test": _assign_row_ids(test, f"{prefix}_test"),
    }
    paths: Dict[str, Path] = {}
    for split, frame in split_frames.items():
        path = output_dir / f"{prefix}_{split}.csv"
        frame.to_csv(path, index=False)
        paths[split] = path
    return paths


def read_base_splits(output_dir: Path, prefix: str) -> Dict[str, pd.DataFrame]:
    output_dir = Path(output_dir)
    return {
        split: pd.read_csv(output_dir / f"{prefix}_{split}.csv")
        for split in ["train", "val", "test"]
    }


### 5.2. Entrenamiento del router y auditoría


In [15]:
def fit_rssi_routing(
    train: pd.DataFrame,
    val: pd.DataFrame,
    test: pd.DataFrame,
    rssi_columns: Sequence[str],
    k_values: Sequence[int],
    seed: int = SEED,
    n_estimators: int = 200,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Define zonas con XY de train y aprende RSSI -> zona para val/test.

    CLUSTER_ORACLE solo se conserva para diagnostico. La columna CLUSTER que
    consumen los modelos es predicha por RSSI en validacion y test.
    """
    pre = RSSIPreprocessor(use_mask=True).fit(train, rssi_columns)
    x_train = pre.transform(train)
    x_val = pre.transform(val)
    x_test = pre.transform(test)
    coords_train = train[TARGET_COLUMNS].to_numpy(dtype=float)
    unique_coords = np.unique(coords_train, axis=0)

    route_parts: List[pd.DataFrame] = []
    diagnostics: List[Dict[str, object]] = []
    for k in sorted(set(int(v) for v in k_values)):
        if k < 2 or k > len(unique_coords):
            continue
        kmeans = KMeans(n_clusters=k, random_state=seed, n_init=20)
        kmeans.fit(unique_coords)
        oracle = {
            "train": kmeans.predict(train[TARGET_COLUMNS].to_numpy(dtype=float)),
            "val": kmeans.predict(val[TARGET_COLUMNS].to_numpy(dtype=float)),
            "test": kmeans.predict(test[TARGET_COLUMNS].to_numpy(dtype=float)),
        }
        gate = ExtraTreesClassifier(
            n_estimators=n_estimators,
            min_samples_leaf=2,
            max_features="sqrt",
            class_weight="balanced",
            random_state=seed + k,
            n_jobs=-1,
        )
        gate.fit(x_train, oracle["train"])
        predicted = {
            "train": oracle["train"],
            "val": gate.predict(x_val),
            "test": gate.predict(x_test),
        }
        probabilities = {
            "train": np.ones(len(train), dtype=float),
            "val": np.max(gate.predict_proba(x_val), axis=1),
            "test": np.max(gate.predict_proba(x_test), axis=1),
        }
        for split, frame in [("train", train), ("val", val), ("test", test)]:
            route_parts.append(
                pd.DataFrame(
                    {
                        "ROW_ID": frame["ROW_ID"].astype(str).to_numpy(),
                        "SPLIT": split,
                        "N_CLUSTERS": k,
                        "CLUSTER": predicted[split].astype(int),
                        "CLUSTER_ORACLE": oracle[split].astype(int),
                        "GATE_CONFIDENCE": probabilities[split].astype(float),
                    }
                )
            )
        diagnostics.append(
            {
                "N_CLUSTERS": k,
                "VAL_GATE_ACCURACY": float(accuracy_score(oracle["val"], predicted["val"])),
                "TEST_GATE_ACCURACY_DIAGNOSTIC_ONLY": float(
                    accuracy_score(oracle["test"], predicted["test"])
                ),
                "VAL_MEAN_CONFIDENCE": float(np.mean(probabilities["val"])),
                "TEST_MEAN_CONFIDENCE": float(np.mean(probabilities["test"])),
                "N_TRAIN_POSITIONS": int(len(unique_coords)),
            }
        )
    if not route_parts:
        raise ValueError("No se genero ninguna configuracion de clustering.")
    return pd.concat(route_parts, ignore_index=True), pd.DataFrame(diagnostics)


def save_routing(
    routes: pd.DataFrame,
    diagnostics: pd.DataFrame,
    output_dir: Path,
    prefix: str,
) -> Tuple[Path, Path]:
    output_dir = Path(output_dir)
    route_path = output_dir / f"{prefix}_routes.csv"
    diagnostics_path = output_dir / f"{prefix}_routing_diagnostics.csv"
    routes.to_csv(route_path, index=False)
    diagnostics.to_csv(diagnostics_path, index=False)
    return route_path, diagnostics_path


def validate_splits(
    train: pd.DataFrame,
    val: pd.DataFrame,
    test: pd.DataFrame,
    position_columns: Sequence[str],
) -> Dict[str, object]:
    ids = [set(frame["ROW_ID"].astype(str)) for frame in [train, val, test]]
    if ids[0] & ids[1] or ids[0] & ids[2] or ids[1] & ids[2]:
        raise AssertionError("ROW_ID se solapa entre particiones.")
    pos = [set(position_key(frame, position_columns)) for frame in [train, val, test]]
    return {
        "rows": {"train": len(train), "val": len(val), "test": len(test)},
        "positions": {"train": len(pos[0]), "val": len(pos[1]), "test": len(pos[2])},
        "position_overlap": {
            "train_val": len(pos[0] & pos[1]),
            "train_test": len(pos[0] & pos[2]),
            "val_test": len(pos[1] & pos[2]),
        },
    }


## 6. Preparación completa de TUJI1


In [16]:
def prepare_tuji1_dataset(
    data_directories: Sequence[Path],
    output_dir: Path,
    k_values: Sequence[int] = tuple(range(2, 11)),
    val_size: float = 0.15,
    seed: int = SEED,
    gate_estimators: int = 200,
) -> Dict[str, object]:
    """Mantiene el test TUJI1 oficial y divide train por posicion 3D."""
    directories = [Path(path) for path in data_directories]
    original_train, source_train = _load_tuji_split(
        "training_data_complete.csv", "TUJI_train_with_clusters.csv", directories
    )
    official_test, source_test = _load_tuji_split(
        "testing_data_complete.csv", "TUJI_test_with_clusters.csv", directories
    )
    required = {"POS_X", "POS_Y", "POS_Z"}
    missing = sorted(required - set(original_train.columns))
    if missing:
        raise ValueError(f"Faltan columnas TUJI1: {missing}")
    rssi = detect_rssi_columns(original_train)
    client_column = "DEVICE" if "DEVICE" in original_train.columns else "DEVICE_LABEL"
    if client_column not in original_train.columns:
        raise ValueError("TUJI1 necesita DEVICE o DEVICE_LABEL para formar clientes.")

    position_columns = ["POS_X", "POS_Y", "POS_Z"]
    groups = position_key(original_train, position_columns)
    splitter = GroupShuffleSplit(n_splits=1, test_size=val_size, random_state=seed)
    train_idx, val_idx = next(splitter.split(original_train, groups=groups))
    train = original_train.iloc[train_idx].reset_index(drop=True)
    val = original_train.iloc[val_idx].reset_index(drop=True)
    test = official_test.reset_index(drop=True).copy()
    for frame in [train, val, test]:
        frame["CLIENT_ID"] = frame[client_column].astype(str)
    train, val, test = [
        _add_local_metric_targets(frame, crs_name="TUJI1_LOCAL_METRES")
        for frame in [train, val, test]
    ]

    paths = save_base_splits(train, val, test, output_dir, "tuji1_official")
    saved = read_base_splits(output_dir, "tuji1_official")
    routes, route_diag = fit_rssi_routing(
        saved["train"], saved["val"], saved["test"], rssi, k_values, seed, gate_estimators
    )
    route_paths = save_routing(routes, route_diag, output_dir, "tuji1_official")
    diagnostics = validate_splits(
        saved["train"], saved["val"], saved["test"], position_columns
    )
    diagnostics["clients"] = {
        split: sorted(frame["CLIENT_ID"].astype(str).unique().tolist())
        for split, frame in saved.items()
    }
    if diagnostics["position_overlap"]["train_test"] != 0:
        raise AssertionError("El test oficial TUJI1 comparte posiciones con train.")
    report = {
        "prefix": "tuji1_official",
        "source": {"train": source_train, "test": source_test},
        "files": {**{k: str(v) for k, v in paths.items()}, "routes": str(route_paths[0])},
        "diagnostics": diagnostics,
        "routing": route_diag.to_dict("records"),
    }
    output_dir = Path(output_dir)
    with (output_dir / "tuji1_official_preparation_report.json").open(
        "w", encoding="utf-8"
    ) as handle:
        json.dump(report, handle, indent=2, ensure_ascii=False)
    return report


## 7. Configuración y ejecución


In [17]:
DATA_DIRS = [
    Path.cwd(),
    ROOT / "TUJI1",
    ROOT / "Dataset",
    ROOT.parent,
    ROOT.parent / "Dataset",
    Path("/mnt/data"),
]
OUTPUT_DIR = ROOT / "prepared" / "TUJI1"

K_VALUES = list(range(2, 11))
VALIDATION_SIZE = 0.15
SEED = 42
GATE_TREES = 200

report = prepare_tuji1_dataset(
    data_directories=DATA_DIRS,
    output_dir=OUTPUT_DIR,
    k_values=K_VALUES,
    val_size=VALIDATION_SIZE,
    seed=SEED,
    gate_estimators=GATE_TREES,
)
PREFIX = report["prefix"]
print("Datos preparados en:", OUTPUT_DIR)
print("Prefijo:", PREFIX)


Datos preparados en: /home/coder/Indoor/Notebooks/prepared/TUJI1
Prefijo: tuji1_official


## 8. Auditoría final


In [18]:
print("\n=== Auditoría de particiones ===")
display(pd.DataFrame([report["diagnostics"]["rows"]], index=["filas"]))
display(pd.DataFrame([report["diagnostics"]["positions"]], index=["posiciones"]))
display(pd.DataFrame([report["diagnostics"]["position_overlap"]], index=["solapamiento"]))
print("Dispositivos:", report["diagnostics"]["clients"])

print("\n=== Calidad del router RSSI ===")
display(pd.DataFrame(report["routing"]))

overlap = report["diagnostics"]["position_overlap"]
assert overlap["train_val"] == 0
assert overlap["train_test"] == 0
assert overlap["val_test"] == 0
print("\nOK: test oficial intacto y ninguna posición compartida.")



=== Auditoría de particiones ===


,train,val,test
filas,5740,1012,2147


,train,val,test
posiciones,1148,203,431


,train_val,train_test,val_test
solapamiento,0,0,0


Dispositivos: {'train': ['1', '2', '3', '4', '5'], 'val': ['1', '2', '3', '4', '5'], 'test': ['1', '2', '3', '4', '5']}

=== Calidad del router RSSI ===


,N_CLUSTERS,VAL_GATE_ACCURACY,TEST_GATE_ACCURACY_DIAGNOSTIC_ONLY,VAL_MEAN_CONFIDENCE,TEST_MEAN_CONFIDENCE,N_TRAIN_POSITIONS
0,2,0.975296,0.976246,0.918498,0.910176,1148
1,3,0.928854,0.924546,0.817010,0.819511,1148
2,4,0.911067,0.913367,0.773001,0.771011,1148
3,5,0.857708,0.834187,0.692439,0.682416,1148
4,6,0.809289,0.805310,0.646449,0.621503,1148
5,7,0.813241,0.790871,0.619415,0.594005,1148
6,8,0.806324,0.743363,0.574006,0.542555,1148
7,9,0.736166,0.742897,0.533060,0.521158,1148
8,10,0.728261,0.717746,0.526578,0.505312,1148



OK: test oficial intacto y ninguna posición compartida.
